# Urban Heat Island (UHI) - Feature Extraction

**Objective:** Extract spectral and building features for each ground-truth UHI measurement point  
in Santiago (Chile), Rio de Janeiro (Brazil), and Freetown (Sierra Leone), ready for model training.

**Pipeline Overview:**

1. **Spectral Feature Extraction** — For each coordinate point, sample a 100m buffer  
   from Sentinel-2 GeoTIFFs and compute 21 features:
   - Raw band medians: B02 (Blue), B03 (Green), B04 (Red), B08 (NIR), B11 (SWIR1)
   - Spectral indices: NDVI, NDBI, NDWI, NDMI, BSI
   - New indices: EVI, SAVI, MNDWI, UI, MSI, dry_surface
   - Within-buffer variability: NDVI_std, NDBI_std, NDWI_std, B08_std, B11_std

2. **Building Footprint Feature Extraction** — For each coordinate point, spatially join  
   to 3D-GloBFP shapefiles and compute:
   - building_density_100m, building_coverage
   - mean_building_height, weighted_height, avg_building_size

3. **Output** — One combined CSV per region containing coordinates, UHI class labels  
   (where available) and all extracted spectral and building features, ready to be  
   passed directly into the modelling notebook.

---

**Inputs:**
- Sentinel-2 GeoTIFF per region (Chile, Brazil, Sierra Leone)
- 3D-GloBFP shapefiles per region (Chile, Brazil, Sierra Leone)
- UHI label CSVs (Chile, Brazil) and validation coordinates (Sierra Leone)

**Outputs:**
- `chile_combined.csv`
- `brazil_combined.csv`
- `sierra_combined.csv`

## 1. Setup

In [ ]:
# install packages
!pip install rioxarray
!pip install pyproj
!pip install rasterio


In [4]:
# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Standard library
import os
import zipfile
from collections import Counter

# Data manipulation
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Geospatial — raster
import xarray as xr
import rioxarray as rxr
import rasterio

# Geospatial — vector
import geopandas as gpd
from shapely.geometry import box, Point
from shapely.strtree import STRtree
from pyproj import Proj, Transformer, CRS

# Statistics
from scipy.stats import ks_2samp

# Progress bar
from tqdm import tqdm

#Unzip building footprints
#Upload the file Building_Footprint_Data.zip to /Data/
with zipfile.ZipFile('./Data/Building_Footprint_Data.zip', 'r') as z:
   z.extractall('./Data')

print("Setup complete.")

Setup complete.


## 2. Configuration
All tunable parameters in one place.

In [9]:
# File paths
PATHS = {
    'chile_uhi':        './Data/sample_chile_uhi_data.csv',
    'brazil_uhi':       './Data/sample_brazil_uhi_data.csv',
    'sierra_uhi':       './Data/validation_dataset.csv',
    'chile_tiff':       './Data/sample_chile.tiff',
    'brazil_tiff':      './Data/sample_Brazil.tiff',
    'sierra_tiff':      './Data/sample_Sierra.tiff',
    'chile_shp':        './Data/Chile Building Footprints/Chile.shp',
    'brazil_shp':       './Data/Brazil Building Footprints/Brazil.shp',
    'sierra_shp':       './Data/Sierra Leone Building Footprints/Sierra Leone.shp',
    'chile_combined':   './Data/chile_combined.csv',
    'brazil_combined':  './Data/Brazil_combined.csv',
    'sierra_combined':  './Data/sierra_combined.csv',
    'submission':       './Data/Predicted_Dataset.csv',
}

# Feature sets
SPECTRAL_COLS = [
    'Longitude', 'Latitude', 'UHI_Class',
    # Raw bands
    'B02', 'B03', 'B04', 'B08', 'B11',
    # Original indices
    'median_NDVI', 'median_NDBI', 'median_NDWI',
    'NDVI_std', 'NDBI_std', 'NDWI_std',
    'NDMI', 'BSI',
    # New indices
    'EVI', 'SAVI', 'MNDWI', 'UI', 'MSI',
    'dry_surface', 'B08_std', 'B11_std',
]

BUILDING_COLS = [
    'building_density_100m', 'building_coverage',
    'avg_building_size', 'mean_building_height', 'weighted_height',
]

MODEL_FEATURES = [
    # Raw bands
    'B02', 'B03', 'B04', 'B08', 'B11',
    # Original indices + stds
    'median_NDVI', 'median_NDBI', 'median_NDWI',
    'NDVI_std', 'NDBI_std', 'NDWI_std',
    'NDMI', 'BSI',
    # New spectral indices
    'EVI', 'SAVI', 'MNDWI', 'UI', 'MSI',
    'dry_surface', 'B08_std', 'B11_std',
    # Building features
    'building_density_100m', 'building_coverage',
    'avg_building_size', 'mean_building_height', 'weighted_height',
]



## 3. Feature Extraction

### 3.1 Spectral Indices from GeoTIFF
Extracts NDVI, NDBI, NDWI (and derived features) using a 100m buffer mean  
around each ground traverse point. Buffer mean is more robust than single-pixel  
sampling as it reduces GPS jitter noise.

In [12]:
def extract_spectral_features(geotiff_path, csv_path, buffer_deg=0.0009):
    """
    Extract spectral indices from a GeoTIFF using a buffer mean around each point.

    Parameters
    ----------
    geotiff_path : str   Path to GeoTIFF (band1=NDVI, band2=NDBI, band3=NDWI)
    csv_path     : str   Path to CSV with Latitude and Longitude columns
    buffer_deg   : float Buffer radius in degrees (~0.0009 deg ≈ 100m at equator)

    Returns
    -------
    pd.DataFrame with spectral feature columns added
    """
    data     = rxr.open_rasterio(geotiff_path)
    df       = pd.read_csv(csv_path)
    x_coords = data.x.values
    y_coords = data.y.values

    results = {k: [] for k in ['NDVI', 'NDBI', 'NDWI', 'NDVI_std', 'NDBI_std', 'NDWI_std', 'NDMI', 'BSI']}

    for lat, lon in tqdm(zip(df['Latitude'].values, df['Longitude'].values),
                         total=len(df), desc=f"Extracting {geotiff_path.split('/')[-1]}"):

        x_mask = (x_coords >= lon - buffer_deg) & (x_coords <= lon + buffer_deg)
        y_mask = (y_coords >= lat - buffer_deg) & (y_coords <= lat + buffer_deg)
        patch  = data.isel(x=np.where(x_mask)[0], y=np.where(y_mask)[0])

        def clean(arr):
            a = arr.flatten().astype(float)
            return a[~np.isnan(a) & (a != 0)]

        ndvi = clean(patch.sel(band=1).values)
        ndbi = clean(patch.sel(band=2).values)
        ndwi = clean(patch.sel(band=3).values)

        ndvi_m = np.mean(ndvi) if len(ndvi) > 0 else 0
        ndbi_m = np.mean(ndbi) if len(ndbi) > 0 else 0
        ndwi_m = np.mean(ndwi) if len(ndwi) > 0 else 0

        results['NDVI'].append(ndvi_m)
        results['NDBI'].append(ndbi_m)
        results['NDWI'].append(ndwi_m)
        results['NDVI_std'].append(np.std(ndvi) if len(ndvi) > 0 else 0)
        results['NDBI_std'].append(np.std(ndbi) if len(ndbi) > 0 else 0)
        results['NDWI_std'].append(np.std(ndwi) if len(ndwi) > 0 else 0)
        results['NDMI'].append(-ndbi_m)  # moisture = inverse of built-up
        bsi = (ndbi_m - ndvi_m) / (ndbi_m + ndvi_m) if (ndbi_m + ndvi_m) != 0 else 0
        results['BSI'].append(np.clip(bsi, -1, 1))  # clip to valid range

    df['median_NDVI'] = results['NDVI']
    df['median_NDBI'] = results['NDBI']
    df['median_NDWI'] = results['NDWI']
    df['NDVI_std']    = results['NDVI_std']
    df['NDBI_std']    = results['NDBI_std']
    df['NDWI_std']    = results['NDWI_std']
    df['NDMI']        = results['NDMI']
    df['BSI']         = results['BSI']

    return df


print("extract_spectral_features() defined.")

extract_spectral_features() defined.


### 3.2 Building Features from Shapefile
Extracts density, coverage, size and height within a 100m buffer.  
Uses city-level clipping and a spatial index (STRtree) for speed.

In [13]:
def extract_building_features(
    csv_path,
    shp_path,
    buffer_m=100,
    margin_m=200,
    height_col='Height'
):
    """
    Extract building features within a square buffer around each point.

    Features extracted
    ------------------
    building_density_100m : buildings per m² in buffer
    building_coverage     : fraction of buffer area covered by buildings
    avg_building_size     : mean building footprint area (m²)
    mean_building_height  : mean building height from Height column
    weighted_height       : height weighted by footprint area (street canyon proxy)

    Parameters
    ----------
    csv_path   : str   Path to CSV with Latitude / Longitude columns
    shp_path   : str   Path to building footprint shapefile
    buffer_m   : float Buffer size in metres (default 100)
    margin_m   : float Extra margin for city-level clipping (default 200)
    height_col : str   Height column name in shapefile (default 'Height')
    """

    # Load points
    df = pd.read_csv(csv_path)
    df['geometry'] = df.apply(
        lambda r: Point(r['Longitude'], r['Latitude']),
        axis=1
    )
    gdf_points = gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326')

    # Load buildings
    print(f"Loading {shp_path.split('/')[-1]}...")
    gdf_bldg = gpd.read_file(shp_path)
    print(f"  Total buildings loaded: {len(gdf_bldg):,}")

    # Validate height column
    if height_col not in gdf_bldg.columns:
        raise ValueError(
            f"Height column '{height_col}' not found. "
            f"Available columns: {gdf_bldg.columns.tolist()}"
        )
    valid_heights = (gdf_bldg[height_col].fillna(0) > 0).sum()
    print(f"  Height column: '{height_col}' ({valid_heights:,} valid values)")

    # Clip buildings to city extent
    # Reduces the number of buildings before reprojection — much faster
    margin_deg = margin_m / 111320
    bounds     = gdf_points.total_bounds   # (minx, miny, maxx, maxy)

    clip_box = box(
        bounds[0] - margin_deg,   # west
        bounds[1] - margin_deg,   # south
        bounds[2] + margin_deg,   # east
        bounds[3] + margin_deg    # north
    )

    gdf_bldg = gdf_bldg[
        gdf_bldg.geometry.intersects(clip_box)
    ].copy().reset_index(drop=True)

    print(f"  Buildings after clipping: {len(gdf_bldg):,}")

    # Reproject to EPSG:3857 for metre-based buffering
    gdf_bldg   = gdf_bldg.to_crs(epsg=3857)
    gdf_points = gdf_points.to_crs(epsg=3857)

    # Build spatial index once
    # STRtree avoids checking every building for every point
    tree = STRtree(gdf_bldg.geometry)

    # Result lists
    densities       = []
    coverages       = []
    avg_sizes       = []
    mean_heights    = []
    weighted_heights = []

    #Main loop
    for _, row in tqdm(
        gdf_points.iterrows(),
        total=len(gdf_points),
        desc="Extracting building features"
    ):
        point_geom = row.geometry

        # Square buffer centred on the point
        buffer_geom = box(
            point_geom.x - buffer_m / 2,
            point_geom.y - buffer_m / 2,
            point_geom.x + buffer_m / 2,
            point_geom.y + buffer_m / 2
        )

        # Find candidate buildings using spatial index
        candidate_idx = tree.query(buffer_geom)

        if len(candidate_idx) == 0:
            buildings_in_buffer = gpd.GeoDataFrame()
        else:
            candidates = gdf_bldg.iloc[candidate_idx]
            buildings_in_buffer = candidates[
                candidates.geometry.intersects(buffer_geom)
            ].copy()
            # Fix any invalid geometries
            buildings_in_buffer['geometry'] = (
                buildings_in_buffer.geometry.buffer(0)
            )

        # Compute features
        area_m2  = buffer_geom.area
        n_bldg   = len(buildings_in_buffer)
        density  = n_bldg / area_m2 if area_m2 > 0 else 0

        if n_bldg > 0:
            footprint_areas = buildings_in_buffer.geometry.area.values
            total_footprint = np.sum(footprint_areas)
            avg_size        = np.mean(footprint_areas)
            coverage        = total_footprint / area_m2

            heights    = buildings_in_buffer[height_col].fillna(0).values.astype(float)
            mean_h     = np.mean(heights)
            weighted_h = (
                np.sum(heights * footprint_areas) / total_footprint
                if total_footprint > 0
                else 0
            )
        else:
            avg_size   = 0
            coverage   = 0
            mean_h     = 0
            weighted_h = 0

        densities.append(density)
        coverages.append(coverage)
        avg_sizes.append(avg_size)
        mean_heights.append(mean_h)
        weighted_heights.append(weighted_h)

    # Add results to GeoDataFrame
    gdf_points['building_density_100m'] = densities
    gdf_points['building_coverage']     = coverages
    gdf_points['avg_building_size']     = avg_sizes
    gdf_points['mean_building_height']  = mean_heights
    gdf_points['weighted_height']       = weighted_heights

    return gdf_points


print("extract_building_features() defined.")

extract_building_features() defined.


### 3.3 Run Feature Extraction

In [ ]:
# Spectral features
sierra_spectral = extract_spectral_features(PATHS['sierra_tiff'], PATHS['sierra_uhi'])
chile_spectral  = extract_spectral_features(PATHS['chile_tiff'],  PATHS['chile_uhi'])
brazil_spectral = extract_spectral_features(PATHS['brazil_tiff'], PATHS['brazil_uhi'])


In [ ]:

# Building features
chile_bldg  = extract_building_features(PATHS['chile_uhi'],  PATHS['chile_shp'])
brazil_bldg = extract_building_features(PATHS['brazil_uhi'], PATHS['brazil_shp'])
sierra_bldg = extract_building_features(PATHS['sierra_uhi'], PATHS['sierra_shp'])

In [ ]:
chile_bldg = pd.read_csv("./Data/chile_buildings_data.csv")
brazil_bldg = pd.read_csv("./Data/Brazil_buildings_data.csv")
sierra_bldg = pd.read_csv("./Data/Sierra_buildings_data.csv")

In [ ]:

# Combine and save
chile_combined  = pd.concat([chile_spectral[SPECTRAL_COLS],  chile_bldg[BUILDING_COLS]],  axis=1)
brazil_combined = pd.concat([brazil_spectral[SPECTRAL_COLS], brazil_bldg[BUILDING_COLS]], axis=1)

# Sierra has no UHI_Class labels — drop from spectral cols
sierra_cols = [c for c in SPECTRAL_COLS if c != 'UHI_Class']
sierra_combined = pd.concat([sierra_spectral[sierra_cols], sierra_bldg[BUILDING_COLS]], axis=1)

chile_combined.to_csv(PATHS['chile_combined'],  index=False)
brazil_combined.to_csv(PATHS['brazil_combined'], index=False)
sierra_combined.to_csv(PATHS['sierra_combined'], index=False)

print("All combined CSVs saved.")
print(f"   Chile:   {chile_combined.shape}")
print(f"   Brazil:  {brazil_combined.shape}")
print(f"   Freetown:{sierra_combined.shape}")

All combined CSVs saved.
   Chile:   (21662, 29)
   Brazil:  (28488, 29)
   Freetown:(14105, 28)
